# Labelled examples

In [1]:
import pandas as pd
import datetime as dt
from src.text_processing_functions import *
from src.LLM_functions import *
import copy as cp

from src.data import *
from src.labelling_helpers import *
from src.post_process_functions import *

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lhasbini\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lhasbini\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lhasbini\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# Load data
from src.data_format import format_output

# old_file = "preproc_filtered_report_types_nat_hazards_bugfix_v250925"
new_file = (
    "preproc_text_sel_gaps_df_with_clean_text_all_v190526"
)

processed_reports_all = pd.read_csv(DATA_IN_JSONS / (new_file + ".csv"))

processed_reports_all = format_output(
    processed_reports_all, list_cols=["sentences", "nathaz_text"]
)
processed_reports_all = take_latest_report(processed_reports_all)

C:\Users\lhasbini\ownCloud\Documents\Thèse\programs\como_project4\como_project4\src\labelling_helpers.py:29: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sort_values(date_field, ascending=False).head(1))


In [ ]:
# #Verify the number of reports which are not in English
# reports_not_english = ['MDRLY005', 'MDRYE014', 'MDRIN012', 'MDRIN009', 'MDRIN009', 'MDRIN009', 'MDRPK006', 'MDRIN006', 'MDRCN003', 'MDRBD002', 'MDRVN003', 'MDRBD002', 'MDRBD002', 'MDRBD002', 'MDRBD002', 'MDRBD002', '05EA022', 'MDRPH016', 'MDR55001']
# processed_reports_all_not_english = processed_reports_all["appealCode"].isin(reports_not_english)

In [3]:
# appealCode_list = ["MDRCN006", "MDRBD022", "MDRYE011", "MDRS2001", "MDRIQ014", "MDRGN015", "MDRSV012", "MDRMY003", "MDRKE058", "MDRNG041", "MDRZM022", "MDRUG050"]
appealCode_list = [
    "MDRUY004",
    "MDRKZ010",
    "MDREC019",
    "MDRPG008",
    "MDRTJ035", 
    "MDRPK018"
]


## Select the reports corresponding to the ones already labelled
#keys = labelled_v1_reports[['appealCode', 'reportDate']].drop_duplicates()
#reports_to_label_1 = filtered_reports.merge(keys, on=['appealCode', 'reportDate'], how='inner')
#appealCode_labelled = reports_to_label_1.appealCode.unique()
#
## For the rest used the previous selection method
#appealCode_select = list(set(appealCode_list) - set(appealCode_labelled))
#reports_to_label_all = filtered_reports.loc[filtered_reports.appealCode.isin(appealCode_select)]
#reports_to_label_2 = take_longest_report(reports_to_label_all)
#
#reports_to_label = pd.concat([reports_to_label_1, reports_to_label_2], ignore_index=True)
#
##check that everything is there
#print(f"Number appealCodes: {len(appealCode_list)}, number reports: {len(reports_to_label)}")
#missing_appealCode = list(set(appealCode_list) - set(reports_to_label.appealCode))
#print(f"Missing appealCodes: {missing_appealCode}")

In [4]:
reports_to_label = processed_reports_all.loc[processed_reports_all.appealCode.isin(appealCode_list)]

In [5]:
reports_to_label

,uid,pdf_text,extraction_error,text,reportName,reportLink,disasterType,origType,dateTime,location,...,naturalHazard,language,iso_code,reportDate,text_processed,nathaz_text,keep_headers,drop_headers,hazards_found_kw,secondaryDisasterType
448,d15db9307397886055ad107497bfec14,Emergency Plan of Action (EPoA) ...,NaN,Emergency Plan of Action (EPoA)\nEcuador: Eart...,MDREC019do.pdf,https://adore.ifrc.org/Download.aspx?FileId=51...,Earthquake,DREF Operation,2022-04-12T00:00:00+02:00,,...,1,en,ECU,2022-04-12,Emergency Plan of Action (EPoA)\nEcuador: Eart...,[Emergency Plan of Action (EPoA) Ecuador: Eart...,{2104: 'A. Situation analysis'},"{8657: 'Summary of the current response', 1715...","['Wildfire', 'Earthquake', 'Mass movement', 'F...",NaN
723,d4f0a8e060c97b0e72e69a5f25489096,DREF Final Report ...,NaN,DREF Final Report\nKazakhstan: Drought\nDREF o...,MDRKZ010dfr.pdf,https://adore.ifrc.org/Download.aspx?FileId=51...,Drought,DREF Operation Final Report,2022-04-01T00:00:00+02:00,,...,1,en,KAZ,2022-04-01,DREF Final Report\nKazakhstan: Drought\nDREF o...,[DREF Final Report Kazakhstan: Drought DREF op...,{671: 'A. Situation analysis'},"{5067: 'Summary of response', 21104: 'Narrativ...","['Drought', 'Wildfire', 'Wave action', 'Extrem...",NaN
985,a1d3c99f697b3d7c92e6d4324e42b0fb,EEmmeerrggeennccyy PPllaann ooff AAccttiioonn ...,NaN,EEmmeerrggeennccyy PPllaann ooff AAccttiioonn ...,MDRPG008dfr.pdf,https://adore.ifrc.org/Download.aspx?FileId=21...,Earthquake,DREF Operation Final Report,2018-11-24T00:00:00+01:00,,...,1,en,PNG,2018-11-24,EEmmeerrggeennccyy PPllaann ooff AAccttiioonn ...,[EEmmeerrggeennccyy PPllaann ooff AAccttiioonn...,{3692: 'A. SITUATION ANALYSIS'},"{6039: 'Summary of current response', 16454: '...","['Earthquake', 'Mass movement', 'Flood', 'Conf...",NaN
1049,af2195a5006a91ec296399ccbb5f54a7,Final Report ...,NaN,Final Report\nPakistan: Severe winter\nDREF op...,MDRPK018dfr.pdf,https://adore.ifrc.org/Download.aspx?FileId=36...,Cold Wave,DREF Operation Final Report,2020-11-30T00:00:00+01:00,,...,1,en,PAK,2020-11-30,Final Report\nPakistan: Severe winter\nDREF op...,[Final Report Pakistan: Severe winter DREF ope...,{1274: 'A. SITUATION ANALYSIS'},"{5321: 'Summary of response', 32526: 'Narrativ...","['Earthquake', 'Mass movement', 'Flood', 'Wave...",NaN
1256,76441a900ae7e3754d2ccfcd0a6c5611,DREF Final Report ...,NaN,DREF Final Report\nTajikistan Earthquake 2023\...,Tajikistan Earthquake 2023 (MDRTJ035),https://go-api.ifrc.org/api/DownloadFile/84135...,Earthquake,MDRTJ035dfr,2025-12-08T11:05:00+01:00,,...,1,en,TJK,2025-12-08,DREF Final Report\nTajikistan Earthquake 2023\...,[DREF Final Report Tajikistan Earthquake 2023 ...,"{1028: 'Description of the Event', 1107: 'What...","{3441: 'National Society Actions', 6020: 'ICRC...",['Earthquake'],NaN
1322,339041cb20bea17ef2945ed9fa55726f,OPERATIONAL UPDATE ...,NaN,OPERATIONAL UPDATE\nUruguay: Droughts - Januar...,Uruguay - Drought (MDRUY004),https://go-api.ifrc.org/api/DownloadFile/84891...,Drought,MDRUY004ou2,2025-12-08T11:05:00+01:00,,...,1,en,URY,2025-12-08,OPERATIONAL UPDATE\nUruguay: Droughts - Januar...,[OPERATIONAL UPDATE Uruguay: Droughts - Januar...,"{579: 'Description of the Event', 656: 'What h...","{11714: 'Current National Society Actions', 18...","['Drought', 'Wildfire', 'Flood', 'Extreme warm...",NaN


In [13]:
# # selection using previous labelling
# # read old label to get report dates
# labelled_reports = pd.read_csv(
#     DATA_LABELLED / "labelled_reports_impacts_all_v111025.csv"
# )
# keys = labelled_reports[["appealCode", "reportDate"]].drop_duplicates()
# reports_to_label = processed_reports_all.merge(
#     keys, on=["appealCode", "reportDate"], how="inner"
# )

In [14]:
# print(
#     f"Labelled reports: {labelled_reports.appealCode.nunique()} ; Selected reports: {reports_to_label.appealCode.nunique()}"
# )

In [32]:
labelled_impact_reports_dict = {} # dict to store labelled reports
#empty dict structure to store results
# labelled_impact_reports_dict["appealCode"]=[
#     {"reportDate": None,
#      "impactSubtype" : None,
#      "impactValue" : None,
#      "impactUnit" : None,
#      "impactValuePrecision" : None,
#      "impactValueMin" : None,
#      "impactValueMax" : None,
#      "annotation" : None,
#      "country" : None,
#      "location" : None,
#      "startYear" : None,
#      "startMonth" : None,
#      "startDay" : None,
#      "endYear" : None,
#      "endMonth" : None,
#      "endDay" : None,
#      "hazards" : None,
#     },
# ]

# MDRUY004

In [16]:
reports_to_label

,uid,pdf_text,extraction_error,text,reportName,reportLink,disasterType,origType,dateTime,location,...,naturalHazard,language,iso_code,reportDate,text_processed,nathaz_text,keep_headers,drop_headers,hazards_found_kw,secondaryDisasterType
448,d15db9307397886055ad107497bfec14,Emergency Plan of Action (EPoA) ...,NaN,Emergency Plan of Action (EPoA)\nEcuador: Eart...,MDREC019do.pdf,https://adore.ifrc.org/Download.aspx?FileId=51...,Earthquake,DREF Operation,2022-04-12T00:00:00+02:00,,...,1,en,ECU,2022-04-12,Emergency Plan of Action (EPoA)\nEcuador: Eart...,[Emergency Plan of Action (EPoA) Ecuador: Eart...,{2104: 'A. Situation analysis'},"{8657: 'Summary of the current response', 1715...","['Wildfire', 'Earthquake', 'Mass movement', 'F...",NaN
723,d4f0a8e060c97b0e72e69a5f25489096,DREF Final Report ...,NaN,DREF Final Report\nKazakhstan: Drought\nDREF o...,MDRKZ010dfr.pdf,https://adore.ifrc.org/Download.aspx?FileId=51...,Drought,DREF Operation Final Report,2022-04-01T00:00:00+02:00,,...,1,en,KAZ,2022-04-01,DREF Final Report\nKazakhstan: Drought\nDREF o...,[DREF Final Report Kazakhstan: Drought DREF op...,{671: 'A. Situation analysis'},"{5067: 'Summary of response', 21104: 'Narrativ...","['Drought', 'Wildfire', 'Wave action', 'Extrem...",NaN
985,a1d3c99f697b3d7c92e6d4324e42b0fb,EEmmeerrggeennccyy PPllaann ooff AAccttiioonn ...,NaN,EEmmeerrggeennccyy PPllaann ooff AAccttiioonn ...,MDRPG008dfr.pdf,https://adore.ifrc.org/Download.aspx?FileId=21...,Earthquake,DREF Operation Final Report,2018-11-24T00:00:00+01:00,,...,1,en,PNG,2018-11-24,EEmmeerrggeennccyy PPllaann ooff AAccttiioonn ...,[EEmmeerrggeennccyy PPllaann ooff AAccttiioonn...,{3692: 'A. SITUATION ANALYSIS'},"{6039: 'Summary of current response', 16454: '...","['Earthquake', 'Mass movement', 'Flood', 'Conf...",NaN
1049,af2195a5006a91ec296399ccbb5f54a7,Final Report ...,NaN,Final Report\nPakistan: Severe winter\nDREF op...,MDRPK018dfr.pdf,https://adore.ifrc.org/Download.aspx?FileId=36...,Cold Wave,DREF Operation Final Report,2020-11-30T00:00:00+01:00,,...,1,en,PAK,2020-11-30,Final Report\nPakistan: Severe winter\nDREF op...,[Final Report Pakistan: Severe winter DREF ope...,{1274: 'A. SITUATION ANALYSIS'},"{5321: 'Summary of response', 32526: 'Narrativ...","['Earthquake', 'Mass movement', 'Flood', 'Wave...",NaN
1256,76441a900ae7e3754d2ccfcd0a6c5611,DREF Final Report ...,NaN,DREF Final Report\nTajikistan Earthquake 2023\...,Tajikistan Earthquake 2023 (MDRTJ035),https://go-api.ifrc.org/api/DownloadFile/84135...,Earthquake,MDRTJ035dfr,2025-12-08T11:05:00+01:00,,...,1,en,TJK,2025-12-08,DREF Final Report\nTajikistan Earthquake 2023\...,[DREF Final Report Tajikistan Earthquake 2023 ...,"{1028: 'Description of the Event', 1107: 'What...","{3441: 'National Society Actions', 6020: 'ICRC...",['Earthquake'],NaN
1322,339041cb20bea17ef2945ed9fa55726f,OPERATIONAL UPDATE ...,NaN,OPERATIONAL UPDATE\nUruguay: Droughts - Januar...,Uruguay - Drought (MDRUY004),https://go-api.ifrc.org/api/DownloadFile/84891...,Drought,MDRUY004ou2,2025-12-08T11:05:00+01:00,,...,1,en,URY,2025-12-08,OPERATIONAL UPDATE\nUruguay: Droughts - Januar...,[OPERATIONAL UPDATE Uruguay: Droughts - Januar...,"{579: 'Description of the Event', 656: 'What h...","{11714: 'Current National Society Actions', 18...","['Drought', 'Wildfire', 'Flood', 'Extreme warm...",NaN


In [17]:
iappeal = "MDRUY004"
ireport = select_report_by_appealCode(iappeal, reports_to_label).iloc[0]
print_report(iappeal, ireport)

MDRUY004: 2025-12-08
OPERATIONAL UPDATE Uruguay: Droughts - January 2023 Multipurpose Cash Transfer Program, Department of Florida, June 2023.
Source: Uruguayan Red Cross.
Appeal: Total DREF Allocation Crisis Category: Hazard: MDRUY004 CHF 381,390 Yellow Drought Glide Number: People Affected: People Targeted: DR-2023-000010-URY 409,115 people 12,000 people Event Onset: Operation Start Date: New Operational end date: Total operating timeframe: Slow 2023-01-29 2023-08-31 7 months Additional Allocation Re- Targeted Areas: Cerro Largo, Florida, Lavalleja, San Jose, Tacuarem- quested bo Page 1 / 25 Description of the Event Areas affected by droughts in Uruguay.
Source: URC.
What happened, where and when?
The lack of rainfall since September 2022 has caused a significant reduction in the availability and access to water in the country, which in turn has been affected by the presence of the La Niña phenomenon in the region and the increase in temperatures during the summer season.
On 20 Janua

In [18]:
labelled_impact_reports_dict[iappeal]=[
    {"reportDate": "2025-12-08",
     "impactSubtype" : "DREF Allocation & Funding requirements",
     "impactValue" : 381390,
     "impactUnit" : "CHF",
     "impactValuePrecision" : "exact",
     "impactValueMin" : None,
     "impactValueMax" : None,
     "annotation" : ['Appeal: Total DREF Allocation Crisis Category: Hazard: MDRUY004 CHF 381,390 Yellow Drought Glide Number: People Affected: People Targeted: DR-2023-000010-URY 409,115 people 12,000 people Event Onset: Operation Start Date: New Operational end date: Total operating timeframe: Slow 2023-01-29 2023-08-31 7 months Additional Allocation Re- Targeted Areas: Cerro Largo, Florida, Lavalleja, San Jose, Tacuarem- quested bo Page 1 / 25 Description of the Event Areas affected by droughts in Uruguay.'],
     "country" : ["Uruguay"],
     "location" : ["Cerro Largo", "Florida", "Lavalleja", "San Jose", "Tacuarem"],
     "startYear" : 2023,
     "startMonth" : 1,
     "startDay" : 29,
     "endYear" : 2023,
     "endMonth" : 8,
     "endDay" : 31,
     "hazards" : ["Drought"]
    },
    {"reportDate": "2025-12-08",
     "impactSubtype" : "Affected People",
     "impactValue" : 409115,
     "impactUnit" : "people",
     "impactValuePrecision" : "exact",
     "impactValueMin" : None,
     "impactValueMax" : None,
     "annotation" : ['Appeal: Total DREF Allocation Crisis Category: Hazard: MDRUY004 CHF 381,390 Yellow Drought Glide Number: People Affected: People Targeted: DR-2023-000010-URY 409,115 people 12,000 people Event Onset: Operation Start Date: New Operational end date: Total operating timeframe: Slow 2023-01-29 2023-08-31 7 months Additional Allocation Re- Targeted Areas: Cerro Largo, Florida, Lavalleja, San Jose, Tacuarem- quested bo Page 1 / 25 Description of the Event Areas affected by droughts in Uruguay.'],
     "country" : ["Uruguay"],
     "location" : ["Cerro Largo", "Florida", "Lavalleja", "San Jose", "Tacuarem"],
     "startYear" : 2023,
     "startMonth" : 1,
     "startDay" : 29,
     "endYear" : 2023,
     "endMonth" : 8,
     "endDay" : 31,
     "hazards" : ["Drought"]
    },
    {"reportDate": "2025-12-08",
     "impactSubtype" : "Targeted People",
     "impactValue" : 12000,
     "impactUnit" : "people",
     "impactValuePrecision" : "exact",
     "impactValueMin" : None,
     "impactValueMax" : None,
     "annotation" : ['Appeal: Total DREF Allocation Crisis Category: Hazard: MDRUY004 CHF 381,390 Yellow Drought Glide Number: People Affected: People Targeted: DR-2023-000010-URY 409,115 people 12,000 people Event Onset: Operation Start Date: New Operational end date: Total operating timeframe: Slow 2023-01-29 2023-08-31 7 months Additional Allocation Re- Targeted Areas: Cerro Largo, Florida, Lavalleja, San Jose, Tacuarem- quested bo Page 1 / 25 Description of the Event Areas affected by droughts in Uruguay.'],
     "country" : ["Uruguay"],
     "location" : ["Cerro Largo", "Florida", "Lavalleja", "San Jose", "Tacuarem"],
     "startYear" : 2023,
     "startMonth" : 1,
     "startDay" : 29,
     "endYear" : 2023,
     "endMonth" : 8,
     "endDay" : 31,
     "hazards" : ["Drought"]
    },
    {"reportDate": "2025-12-08",
     "impactSubtype" : "Access to Water, Sanitation, and Hygiene",
     "impactValue" : None,
     "impactUnit" : None,
     "impactValuePrecision" : None,
     "impactValueMin" : None,
     "impactValueMax" : None,
     "annotation" : ["Currently, according to information provided by SINAE, the most complex situation is related to the Page 2 / 25 lack of access to safe water which limits families' consumption and use of water (2)."],
     "country" : ["Uruguay"],
     "location" : ["Cerro Largo", "Florida", "Lavalleja", "San Jose", "Tacuarem"],
     "startYear" : 2023,
     "startMonth" : 1,
     "startDay" : 29,
     "endYear" : 2023,
     "endMonth" : 8,
     "endDay" : 31,
     "hazards" : ["Drought"]
    },
    {"reportDate": "2025-12-08",
        "impactSubtype" : "Access to Water, Sanitation, and Hygiene",
        "impactValue" : "2300000",
        "impactUnit" : "people",
        "impactValuePrecision" : "approx",
        "impactValueMin" : None,
        "impactValueMax" : None,
        "annotation" : ["Through the assessment, which included personal interviews and collection of secondary information provided by the State and the official press, it became evident that around 2.3 million people are affected by the inability to regularly access water."],
        "country" : ["Uruguay"],
        "location" : ["Uruguay"],
        "startYear" : 2023,
        "startMonth" : 1,
        "startDay" : 29,
        "endYear" : 2023,
        "endMonth" : 8,
        "endDay" : 31,
        "hazards" : ["Drought"]
    },
    {"reportDate": "2025-12-08",
        "impactSubtype" : "Access to Water, Sanitation, and Hygiene",
        "impactValue" : 409115,
        "impactUnit" : "people",
        "impactValuePrecision" : "approx",
        "impactValueMin" : None,
        "impactValueMax" : None,
        "annotation" : ["In addition, it was estimated that approximately 409,115 people live in areas highly affected by drought and water scarcity, of which about 20,000 have unmet needs related to water, sanitation and hygiene services and livelihoods, such as: general access to water (32%), animal feed (12%), debt financing (10%), water storage inputs (10%), financing for well construction or procurement of water storage inputs (10%), and other water storage inputs (10%)."],
        "country" : ["Uruguay"],
        "location" : ["Uruguay"],
        "startYear" : 2023,
        "startMonth" : 1,
        "startDay" : 29,
        "endYear" : 2023,
        "endMonth" : 8,
        "endDay" : 31,
        "hazards" : ["Drought"]
    },
    {"reportDate": "2025-12-08",
        "impactSubtype" : "Access to Water, Sanitation, and Hygiene",
        "impactValue" : 20000,
        "impactUnit" : "people",
        "impactValuePrecision" : "approx",
        "impactValueMin" : None,
        "impactValueMax" : None,
        "annotation" : ["In addition, it was estimated that approximately 409,115 people live in areas highly affected by drought and water scarcity, of which about 20,000 have unmet needs related to water, sanitation and hygiene services and livelihoods, such as: general access to water (32%), animal feed (12%), debt financing (10%), water storage inputs (10%), financing for well construction or procurement of water storage inputs (10%), and other water storage inputs (10%)."],
        "country" : ["Uruguay"],
        "location" : ["Uruguay"],
        "startYear" : 2023,
        "startMonth" : 1,
        "startDay" : 29,
        "endYear" : 2023,
        "endMonth" : 8,
        "endDay" : 31,
        "hazards" : ["Drought"]
    },
    {"reportDate": "2025-12-08",
        "impactSubtype" : "Economy and Livelihood",
        "impactValue" : None,
        "impactUnit" : None,
        "impactValuePrecision" : None,
        "impactValueMin" : None,
        "impactValueMax" : None,
        "annotation" : ["In addition, it was estimated that approximately 409,115 people live in areas highly affected by drought and water scarcity, of which about 20,000 have unmet needs related to water, sanitation and hygiene services and livelihoods, such as: general access to water (32%), animal feed (12%), debt financing (10%), water storage inputs (10%), financing for well construction or procurement of water storage inputs (10%), and other water storage inputs (10%)."],
        "country" : ["Uruguay"],
        "location" : ["Uruguay"],
        "startYear" : 2023,
        "startMonth" : 1,
        "startDay" : 29,
        "endYear" : 2023,
        "endMonth" : 8,
        "endDay" : 31,
        "hazards" : ["Drought"]
    },
    {"reportDate": "2025-12-08",
        "impactSubtype" : "Affected Livestock and Animals",
        "impactValue" : None,
        "impactUnit" : None,
        "impactValuePrecision" : None,
        "impactValueMin" : None,
        "impactValueMax" : None,
        "annotation" : ["In addition, it was estimated that approximately 409,115 people live in areas highly affected by drought and water scarcity, of which about 20,000 have unmet needs related to water, sanitation and hygiene services and livelihoods, such as: general access to water (32%), animal feed (12%), debt financing (10%), water storage inputs (10%), financing for well construction or procurement of water storage inputs (10%), and other water storage inputs (10%)."],
        "country" : ["Uruguay"],
        "location" : ["Uruguay"],
        "startYear" : 2023,
        "startMonth" : 1,
        "startDay" : 29,
        "endYear" : 2023,
        "endMonth" : 8,
        "endDay" : 31,
        "hazards" : ["Drought"]
    },
]

In [19]:
df_impact = pd.DataFrame(labelled_impact_reports_dict[iappeal])
df_impact['appealCode'] = iappeal
df_impact.reset_index(inplace=True, drop=True)

#Save impact csv
fn = f"labelled_impact_{iappeal}.csv"
#df_impact.to_csv(DATA_LABELLED+fn, index=False)

# MDRKZ010

In [21]:
iappeal = "MDRKZ010"
ireport = select_report_by_appealCode(iappeal, reports_to_label).iloc[0]
print_report(iappeal, ireport)

MDRKZ010: 2022-04-01
DREF Final Report Kazakhstan: Drought DREF operation final report Operation n° MDRKZ010 Date of issue: 1 April 2022 Glide number: DR-2021-000085-KAZ Operation start date: 27 July 2021 Operation end date: 31 December 2021 Host National Society: Red Crescent Society of Operation budget: CHF 497,168 Kazakhstan Number of people assisted: 5,790 Number of people affected: 71,000 (13,000 people reached indirectly through health promotion) Red Cross Red Crescent Movement partners currently actively involved in the operation: UAE Red Crescent Other partner organizations involved in the operation: Government of the Republic of Kazakhstan, local authorities and various NGOs A.
Situation analysis Description of the disaster In Kazakhstan, the heat wave that began in June 2021 in the Southern and Western regions of the country (Kyzylorda, Mangystau and Turkestan regions) led to record temperatures of 46.5°C (recorded on 7 July 2021), with a base average of 28.3°C.
As a result o

In [22]:
labelled_impact_reports_dict[iappeal] = [
    {
        "reportDate": "2022-04-01",
        "impactSubtype": "DREF Allocation & Funding requirements",
        "impactValue": 497168,
        "impactUnit": "CHF",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["DREF Final Report Kazakhstan: Drought DREF operation final report Operation n° MDRKZ010 Date of issue: 1 April 2022 Glide number: DR-2021-000085-KAZ Operation start date: 27 July 2021 Operation end date: 31 December 2021 Host National Society: Red Crescent Society of Operation budget: CHF 497,168 Kazakhstan Number of people assisted: 5,790 Number of people affected: 71,000 (13,000 people reached indirectly through health promotion) Red Cross Red Crescent Movement partners currently actively involved in the operation: UAE Red Crescent Other partner organizations involved in the operation: Government of the Republic of Kazakhstan, local authorities and various NGOs A."],
        "country": ["Kazakhstan"],
        "location": ["Kyzylorda", "Mangystau", "Turkestan"],
        "startYear": 2021,
        "startMonth": 6,
        "startDay": 1,
        "endYear": None,
        "endMonth": None,
        "endDay": None,
        "hazards": ["Drought", "Extreme warm temperature"],
    },
    {
        "reportDate": "2022-04-01",
        "impactSubtype": "Assisted People",
        "impactValue": 5790,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["DREF Final Report Kazakhstan: Drought DREF operation final report Operation n° MDRKZ010 Date of issue: 1 April 2022 Glide number: DR-2021-000085-KAZ Operation start date: 27 July 2021 Operation end date: 31 December 2021 Host National Society: Red Crescent Society of Operation budget: CHF 497,168 Kazakhstan Number of people assisted: 5,790 Number of people affected: 71,000 (13,000 people reached indirectly through health promotion) Red Cross Red Crescent Movement partners currently actively involved in the operation: UAE Red Crescent Other partner organizations involved in the operation: Government of the Republic of Kazakhstan, local authorities and various NGOs A."],
        "country": ["Kazakhstan"],
        "location": ["Kyzylorda", "Mangystau", "Turkestan"],
        "startYear": 2021,
        "startMonth": 6,
        "startDay": 1,
        "endYear": None,
        "endMonth": None,
        "endDay": None,
        "hazards": ["Drought", "Extreme warm temperature"],
    },
    {
        "reportDate": "2022-04-01",
        "impactSubtype": "Affected People",
        "impactValue": 71000,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["DREF Final Report Kazakhstan: Drought DREF operation final report Operation n° MDRKZ010 Date of issue: 1 April 2022 Glide number: DR-2021-000085-KAZ Operation start date: 27 July 2021 Operation end date: 31 December 2021 Host National Society: Red Crescent Society of Operation budget: CHF 497,168 Kazakhstan Number of people assisted: 5,790 Number of people affected: 71,000 (13,000 people reached indirectly through health promotion) Red Cross Red Crescent Movement partners currently actively involved in the operation: UAE Red Crescent Other partner organizations involved in the operation: Government of the Republic of Kazakhstan, local authorities and various NGOs A."],
        "country": ["Kazakhstan"],
        "location": ["Kyzylorda", "Mangystau", "Turkestan"],
        "startYear": 2021,
        "startMonth": 6,
        "startDay": 1,
        "endYear": None,
        "endMonth": None,
        "endDay": None,
        "hazards": ["Drought", "Extreme warm temperature"],
    },
    {
        "reportDate": "2022-04-01",
        "impactSubtype": "Access to Healthcare",
        "impactValue": 13000,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["DREF Final Report Kazakhstan: Drought DREF operation final report Operation n° MDRKZ010 Date of issue: 1 April 2022 Glide number: DR-2021-000085-KAZ Operation start date: 27 July 2021 Operation end date: 31 December 2021 Host National Society: Red Crescent Society of Operation budget: CHF 497,168 Kazakhstan Number of people assisted: 5,790 Number of people affected: 71,000 (13,000 people reached indirectly through health promotion) Red Cross Red Crescent Movement partners currently actively involved in the operation: UAE Red Crescent Other partner organizations involved in the operation: Government of the Republic of Kazakhstan, local authorities and various NGOs A."],
        "country": ["Kazakhstan"],
        "location": ["Kyzylorda", "Mangystau", "Turkestan"],
        "startYear": 2021,
        "startMonth": 6,
        "startDay": 1,
        "endYear": None,
        "endMonth": None,
        "endDay": None,
        "hazards": ["Drought", "Extreme warm temperature"],
    },
    {
        "reportDate": "2022-04-01",
        "impactSubtype": "Affected Livestock and Animals",
        "impactValue": None,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["Due to the inability to graze livestock, the minimum reserves of feed and water were exhausted, which led to the mass death of animals.", 
                       "In addition, agricultural crops were also destroyed due to the heat, which could potentially lead to a serious food crisis in several regions of the country, where cattle are a key and essential livelihood activity due to the geographical and climatic features of the southern and western parts of Kazakhstan.", 
                       "The death of livestock and crops in three regions of the Republic of Kazakhstan (Mangystau, Kyzylorda and Turkestan regions) has had serious impact on the local population since animal husbandry is the only source of income and is a vital activity."],
        "country": ["Kazakhstan"],
        "location": ["Kyzylorda", "Mangystau", "Turkestan"],
        "startYear": 2021,
        "startMonth": 6,
        "startDay": 1,
        "endYear": None,
        "endMonth": None,
        "endDay": None,
        "hazards": ["Drought", "Extreme warm temperature"],
    },
    {
        "reportDate": "2022-04-01",
        "impactSubtype": "Affected Livestock and Animals",
        "impactValue": 2000,
        "impactUnit": "cattle",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["In the affected regions, the death of more than 2,000 cattle was recorded, and this figure was increasing every day."],
        "country": ["Kazakhstan"],
        "location": ["Kyzylorda", "Mangystau", "Turkestan"],
        "startYear": 2021,
        "startMonth": 6,
        "startDay": 1,
        "endYear": None,
        "endMonth": None,
        "endDay": None,
        "hazards": ["Drought", "Extreme warm temperature"],
    },
    {
        "reportDate": "2022-04-01",
        "impactSubtype": "Crop Production and Forestry",
        "impactValue": None,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["In addition, agricultural crops were also destroyed due to the heat, which could potentially lead to a serious food crisis in several regions of the country, where cattle are a key and essential livelihood activity due to the geographical and climatic features of the southern and western parts of Kazakhstan.", 
                       "The death of livestock and crops in three regions of the Republic of Kazakhstan (Mangystau, Kyzylorda and Turkestan regions) has had serious impact on the local population since animal husbandry is the only source of income and is a vital activity."],
        "country": ["Kazakhstan"],
        "location": ["Kyzylorda", "Mangystau", "Turkestan"],
        "startYear": 2021,
        "startMonth": 6,
        "startDay": 1,
        "endYear": None,
        "endMonth": None,
        "endDay": None,
        "hazards": ["Drought", "Extreme warm temperature"],
    },
    {
        "reportDate": "2022-04-01",
        "impactSubtype": "Access to Food",
        "impactValue": None,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["In addition, agricultural crops were also destroyed due to the heat, which could potentially lead to a serious food crisis in several regions of the country, where cattle are a key and essential livelihood activity due to the geographical and climatic features of the southern and western parts of Kazakhstan.", 
                       "Photo from open sources (region unknown) Public The drought greatly affected the food security of the regions, which led to cheaper livestock and higher prices for feed, food and drinking water, which are all critical to ensure the livelihoods of the population.", 
                       "Therefore, the negative consequences due to the sharp deterioration of the socio-economic situation, was the lack of adequate nutrition."],
        "country": ["Kazakhstan"],
        "location": ["Kyzylorda", "Mangystau", "Turkestan"],
        "startYear": 2021,
        "startMonth": 6,
        "startDay": 1,
        "endYear": None,
        "endMonth": None,
        "endDay": None,
        "hazards": ["Drought", "Extreme warm temperature"],
    },
    {
        "reportDate": "2022-04-01",
        "impactSubtype": "Economy and Livelihood",
        "impactValue": None,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["In addition, agricultural crops were also destroyed due to the heat, which could potentially lead to a serious food crisis in several regions of the country, where cattle are a key and essential livelihood activity due to the geographical and climatic features of the southern and western parts of Kazakhstan.", 
                       "Photo from open sources (region unknown) Public The drought greatly affected the food security of the regions, which led to cheaper livestock and higher prices for feed, food and drinking water, which are all critical to ensure the livelihoods of the population.", 
                       "Therefore, the negative consequences due to the sharp deterioration of the socio-economic situation, was the lack of adequate nutrition.", 
                       "The death of livestock and crops in three regions of the Republic of Kazakhstan (Mangystau, Kyzylorda and Turkestan regions) has had serious impact on the local population since animal husbandry is the only source of income and is a vital activity."],
        "country": ["Kazakhstan"],
        "location": ["Kyzylorda", "Mangystau", "Turkestan"],
        "startYear": 2021,
        "startMonth": 6,
        "startDay": 1,
        "endYear": None,
        "endMonth": None,
        "endDay": None,
        "hazards": ["Drought", "Extreme warm temperature"],
    },
    {
        "reportDate": "2022-04-01",
        "impactSubtype": "Water Quality and Availability",
        "impactValue": None,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["Photo from open sources (region unknown) Public The drought greatly affected the food security of the regions, which led to cheaper livestock and higher prices for feed, food and drinking water, which are all critical to ensure the livelihoods of the population."],
        "country": ["Kazakhstan"],
        "location": ["Kyzylorda", "Mangystau", "Turkestan"],
        "startYear": 2021,
        "startMonth": 6,
        "startDay": 1,
        "endYear": None,
        "endMonth": None,
        "endDay": None,
        "hazards": ["Drought", "Extreme warm temperature"],
    },
    {
        "reportDate": "2022-04-01",
        "impactSubtype": "Crop Production and Forestry",
        "impactValue": None,
        "impactUnit": "hectares",
        "impactValuePrecision": "approx",
        "impactValueMin": 1000,
        "impactValueMax": None,
        "annotation": ["In 2021, more than 1,000 hectares of land caught fire on the territory of the Karaganda region due to drought, which, in turn, led to the death of one person and 200 heads of cattle."],
        "country": ["Kazakhstan"],
        "location": ["Karaganda"],
        "startYear": 2021,
        "startMonth": 6,
        "startDay": 1,
        "endYear": 2021,
        "endMonth": 12,
        "endDay": 1,
        "hazards": ["Drought", "Extreme warm temperature", "Wildfire"],
    },
    {
        "reportDate": "2022-04-01",
        "impactSubtype": "Human Deaths",
        "impactValue": 1,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["In 2021, more than 1,000 hectares of land caught fire on the territory of the Karaganda region due to drought, which, in turn, led to the death of one person and 200 heads of cattle."],
        "country": ["Kazakhstan"],
        "location": ["Karaganda"],
        "startYear": 2021,
        "startMonth": 6,
        "startDay": 1,
        "endYear": 2021,
        "endMonth": 12,
        "endDay": 1,
        "hazards": ["Drought", "Extreme warm temperature", "Wildfire"],
    },
    {
        "reportDate": "2022-04-01",
        "impactSubtype": "Affected Livestock and Animals",
        "impactValue": 200,
        "impactUnit": "heads of cattle",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["In 2021, more than 1,000 hectares of land caught fire on the territory of the Karaganda region due to drought, which, in turn, led to the death of one person and 200 heads of cattle."],
        "country": ["Kazakhstan"],
        "location": ["Karaganda"],
        "startYear": 2021,
        "startMonth": 6,
        "startDay": 1,
        "endYear": 2021,
        "endMonth": 12,
        "endDay": 1,
        "hazards": ["Drought", "Extreme warm temperature", "Wildfire"],
    },

]

In [23]:
df_impact = pd.DataFrame(labelled_impact_reports_dict[iappeal])
df_impact['appealCode'] = iappeal
df_impact.reset_index(inplace=True, drop=True)

#Save impact csv
fn = f"labelled_impact_{iappeal}.csv"
#df_impact.to_csv(DATA_LABELLED+fn, index=False)


# MDREC019

In [25]:
iappeal = "MDREC019"
ireport = select_report_by_appealCode(iappeal, reports_to_label).iloc[0]
print_report(iappeal, ireport)

MDREC019: 2022-04-12
Emergency Plan of Action (EPoA) Ecuador: Earthquake DREF Operation MDREC019 Glide n°: EQ-2022-000194-ECU Expected 3 months (From 7 April timeframe: 2022) Date of issue: 12 April 2022 Expected end 31 July 2022 date: Category allocated to the disaster or crisis: Yellow DREF allocated: CHF 167,716 Number of 7,802 people 2,500 people Total number of people affected: people to be (1,560 families) (500 families) assisted: Cotopaxi, Esmeraldas, Guayas, Imbabura, Los Provinces/Re Provinces affected: Ríos, Manabí, Pichincha, gions Esmeraldas Santo Domingo de los targeted: Tsáchilas, Tungurahua.
Host National Society(ies) presence (n° of volunteers, staff, branches): The Ecuadorian Red Cross (ERC) is present in 24 provinces in Ecuador through 24 province branches and 83 canton branches.
It has 7,000 volunteers registered in the national database and 200 staff specialized in different lines of action.
Red Cross Red Crescent Movement partners actively involved in the operation

In [26]:
labelled_impact_reports_dict[iappeal] = [
    {
        "reportDate": "2022-04-12",
        "impactSubtype": "DREF Allocation & Funding requirements",
        "impactValue": 167716,
        "impactUnit": "CHF",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["Emergency Plan of Action (EPoA) Ecuador: Earthquake DREF Operation MDREC019 Glide n°: EQ-2022-000194-ECU Expected 3 months (From 7 April timeframe: 2022) Date of issue: 12 April 2022 Expected end 31 July 2022 date: Category allocated to the disaster or crisis: Yellow DREF allocated: CHF 167,716 Number of 7,802 people 2,500 people Total number of people affected: people to be (1,560 families) (500 families) assisted: Cotopaxi, Esmeraldas, Guayas, Imbabura, Los Provinces/Re Provinces affected: Ríos, Manabí, Pichincha, gions Esmeraldas Santo Domingo de los targeted: Tsáchilas, Tungurahua."],
        "country": ["Ecuador"],
        "location": ["Cotopaxi", "Esmeraldas", "Guayas", "Imbabura", "Los Ríos", "Manabí", "Pichincha", "Santo Domingo de los Tsáchilas", "Tungurahua"],
        "startYear": 2022,
        "startMonth": 3,
        "startDay": 26,
        "endYear": 2022,
        "endMonth": 7,
        "endDay": 31,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2022-04-12",
        "impactSubtype": "Affected people",
        "impactValue": 7802,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["Emergency Plan of Action (EPoA) Ecuador: Earthquake DREF Operation MDREC019 Glide n°: EQ-2022-000194-ECU Expected 3 months (From 7 April timeframe: 2022) Date of issue: 12 April 2022 Expected end 31 July 2022 date: Category allocated to the disaster or crisis: Yellow DREF allocated: CHF 167,716 Number of 7,802 people 2,500 people Total number of people affected: people to be (1,560 families) (500 families) assisted: Cotopaxi, Esmeraldas, Guayas, Imbabura, Los Provinces/Re Provinces affected: Ríos, Manabí, Pichincha, gions Esmeraldas Santo Domingo de los targeted: Tsáchilas, Tungurahua."],
        "country": ["Ecuador"],
        "location": ["Cotopaxi", "Esmeraldas", "Guayas", "Imbabura", "Los Ríos", "Manabí", "Pichincha", "Santo Domingo de los Tsáchilas", "Tungurahua"],
        "startYear": 2022,
        "startMonth": 3,
        "startDay": 26,
        "endYear": 2022,
        "endMonth": 7,
        "endDay": 31,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2022-04-12",
        "impactSubtype": "Assisted people",
        "impactValue": 2500,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["Emergency Plan of Action (EPoA) Ecuador: Earthquake DREF Operation MDREC019 Glide n°: EQ-2022-000194-ECU Expected 3 months (From 7 April timeframe: 2022) Date of issue: 12 April 2022 Expected end 31 July 2022 date: Category allocated to the disaster or crisis: Yellow DREF allocated: CHF 167,716 Number of 7,802 people 2,500 people Total number of people affected: people to be (1,560 families) (500 families) assisted: Cotopaxi, Esmeraldas, Guayas, Imbabura, Los Provinces/Re Provinces affected: Ríos, Manabí, Pichincha, gions Esmeraldas Santo Domingo de los targeted: Tsáchilas, Tungurahua."],
        "country": ["Ecuador"],
        "location": ["Cotopaxi", "Esmeraldas", "Guayas", "Imbabura", "Los Ríos", "Manabí", "Pichincha", "Santo Domingo de los Tsáchilas", "Tungurahua"],
        "startYear": 2022,
        "startMonth": 3,
        "startDay": 26,
        "endYear": 2022,
        "endMonth": 7,
        "endDay": 31,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2022-04-12",
        "impactSubtype": "Affected people",
        "impactValue": 4097,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["17 issued on 6 April 2022 by SNGRE2, the main damages registered include: Summary of People affected and damages Estimate of people affected People affected 4,097 (819 families) People displaced3 3,705 (741 families) People injured 2 Deceased 1 Estimate of infrastructure damaged Houses with slight damage 3,300 Houses with moderate damage 1,428 Houses with severe damage 533 Health centers affected 15 Education centers affected 16 Government buildings affected 12 Private assets affected 6 Bridges affected 1 • Two emergency collective centres were opened by local authorities to provide comprehensive care to people with damaged homes: Table 1: Emergency collective centres opened The rest of affected families have been taken in by friends and families, while others have chosen to build informal shelters made of plastic sheeting and wood next to their homes to guard their belongings."],
        "country": ["Ecuador"],
        "location": ["Esmeralda", "Tabiazo", "Tachina", "San Mateo", "Vuelta Larga", "Chinca", "Camarones", "Quingüe", "Daule de Muisne", "Súa Atacames", "Cotopaxi", "Guayas", "Imbabura", "Los Ríos", "Manabí", "Pichincha", "Santo Domingo de los Tsáchilas", "Tungurahua"],
        "startYear": 2022,
        "startMonth": 3,
        "startDay": 26,
        "endYear": 2022,
        "endMonth": 4,
        "endDay": 6,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2022-04-12",
        "impactSubtype": "Displaced people",
        "impactValue": 3705,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["17 issued on 6 April 2022 by SNGRE2, the main damages registered include: Summary of People affected and damages Estimate of people affected People affected 4,097 (819 families) People displaced3 3,705 (741 families) People injured 2 Deceased 1 Estimate of infrastructure damaged Houses with slight damage 3,300 Houses with moderate damage 1,428 Houses with severe damage 533 Health centers affected 15 Education centers affected 16 Government buildings affected 12 Private assets affected 6 Bridges affected 1 • Two emergency collective centres were opened by local authorities to provide comprehensive care to people with damaged homes: Table 1: Emergency collective centres opened The rest of affected families have been taken in by friends and families, while others have chosen to build informal shelters made of plastic sheeting and wood next to their homes to guard their belongings."],
        "country": ["Ecuador"],
        "location": ["Esmeralda", "Tabiazo", "Tachina", "San Mateo", "Vuelta Larga", "Chinca", "Camarones", "Quingüe", "Daule de Muisne", "Súa Atacames", "Cotopaxi", "Guayas", "Imbabura", "Los Ríos", "Manabí", "Pichincha", "Santo Domingo de los Tsáchilas", "Tungurahua"],
        "startYear": 2022,
        "startMonth": 3,
        "startDay": 26,
        "endYear": 2022,
        "endMonth": 4,
        "endDay": 6,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2022-04-12",
        "impactSubtype": "Injured people",
        "impactValue": 2,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["17 issued on 6 April 2022 by SNGRE2, the main damages registered include: Summary of People affected and damages Estimate of people affected People affected 4,097 (819 families) People displaced3 3,705 (741 families) People injured 2 Deceased 1 Estimate of infrastructure damaged Houses with slight damage 3,300 Houses with moderate damage 1,428 Houses with severe damage 533 Health centers affected 15 Education centers affected 16 Government buildings affected 12 Private assets affected 6 Bridges affected 1 • Two emergency collective centres were opened by local authorities to provide comprehensive care to people with damaged homes: Table 1: Emergency collective centres opened The rest of affected families have been taken in by friends and families, while others have chosen to build informal shelters made of plastic sheeting and wood next to their homes to guard their belongings."],
        "country": ["Ecuador"],
        "location": ["Esmeralda", "Tabiazo", "Tachina", "San Mateo", "Vuelta Larga", "Chinca", "Camarones", "Quingüe", "Daule de Muisne", "Súa Atacames", "Cotopaxi", "Guayas", "Imbabura", "Los Ríos", "Manabí", "Pichincha", "Santo Domingo de los Tsáchilas", "Tungurahua"],
        "startYear": 2022,
        "startMonth": 3,
        "startDay": 26,
        "endYear": 2022,
        "endMonth": 4,
        "endDay": 6,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2022-04-12",
        "impactSubtype": "Human Deaths",
        "impactValue": 1,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["17 issued on 6 April 2022 by SNGRE2, the main damages registered include: Summary of People affected and damages Estimate of people affected People affected 4,097 (819 families) People displaced3 3,705 (741 families) People injured 2 Deceased 1 Estimate of infrastructure damaged Houses with slight damage 3,300 Houses with moderate damage 1,428 Houses with severe damage 533 Health centers affected 15 Education centers affected 16 Government buildings affected 12 Private assets affected 6 Bridges affected 1 • Two emergency collective centres were opened by local authorities to provide comprehensive care to people with damaged homes: Table 1: Emergency collective centres opened The rest of affected families have been taken in by friends and families, while others have chosen to build informal shelters made of plastic sheeting and wood next to their homes to guard their belongings."],
        "country": ["Ecuador"],
        "location": ["Esmeralda", "Tabiazo", "Tachina", "San Mateo", "Vuelta Larga", "Chinca", "Camarones", "Quingüe", "Daule de Muisne", "Súa Atacames", "Cotopaxi", "Guayas", "Imbabura", "Los Ríos", "Manabí", "Pichincha", "Santo Domingo de los Tsáchilas", "Tungurahua"],
        "startYear": 2022,
        "startMonth": 3,
        "startDay": 26,
        "endYear": 2022,
        "endMonth": 4,
        "endDay": 6,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2022-04-12",
        "impactSubtype": "Residential Buildings",
        "impactValue": 3300,
        "impactUnit": "houses with slight damage",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["17 issued on 6 April 2022 by SNGRE2, the main damages registered include: Summary of People affected and damages Estimate of people affected People affected 4,097 (819 families) People displaced3 3,705 (741 families) People injured 2 Deceased 1 Estimate of infrastructure damaged Houses with slight damage 3,300 Houses with moderate damage 1,428 Houses with severe damage 533 Health centers affected 15 Education centers affected 16 Government buildings affected 12 Private assets affected 6 Bridges affected 1 • Two emergency collective centres were opened by local authorities to provide comprehensive care to people with damaged homes: Table 1: Emergency collective centres opened The rest of affected families have been taken in by friends and families, while others have chosen to build informal shelters made of plastic sheeting and wood next to their homes to guard their belongings."],
        "country": ["Ecuador"],
        "location": ["Esmeralda", "Tabiazo", "Tachina", "San Mateo", "Vuelta Larga", "Chinca", "Camarones", "Quingüe", "Daule de Muisne", "Súa Atacames", "Cotopaxi", "Guayas", "Imbabura", "Los Ríos", "Manabí", "Pichincha", "Santo Domingo de los Tsáchilas", "Tungurahua"],
        "startYear": 2022,
        "startMonth": 3,
        "startDay": 26,
        "endYear": 2022,
        "endMonth": 4,
        "endDay": 6,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2022-04-12",
        "impactSubtype": "Residential Buildings",
        "impactValue": 1428,
        "impactUnit": "houses with moderate damage",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["17 issued on 6 April 2022 by SNGRE2, the main damages registered include: Summary of People affected and damages Estimate of people affected People affected 4,097 (819 families) People displaced3 3,705 (741 families) People injured 2 Deceased 1 Estimate of infrastructure damaged Houses with slight damage 3,300 Houses with moderate damage 1,428 Houses with severe damage 533 Health centers affected 15 Education centers affected 16 Government buildings affected 12 Private assets affected 6 Bridges affected 1 • Two emergency collective centres were opened by local authorities to provide comprehensive care to people with damaged homes: Table 1: Emergency collective centres opened The rest of affected families have been taken in by friends and families, while others have chosen to build informal shelters made of plastic sheeting and wood next to their homes to guard their belongings."],
        "country": ["Ecuador"],
        "location": ["Esmeralda", "Tabiazo", "Tachina", "San Mateo", "Vuelta Larga", "Chinca", "Camarones", "Quingüe", "Daule de Muisne", "Súa Atacames", "Cotopaxi", "Guayas", "Imbabura", "Los Ríos", "Manabí", "Pichincha", "Santo Domingo de los Tsáchilas", "Tungurahua"],
        "startYear": 2022,
        "startMonth": 3,
        "startDay": 26,
        "endYear": 2022,
        "endMonth": 4,
        "endDay": 6,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2022-04-12",
        "impactSubtype": "Residential Buildings",
        "impactValue": 533,
        "impactUnit": "houses with severe damage",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["17 issued on 6 April 2022 by SNGRE2, the main damages registered include: Summary of People affected and damages Estimate of people affected People affected 4,097 (819 families) People displaced3 3,705 (741 families) People injured 2 Deceased 1 Estimate of infrastructure damaged Houses with slight damage 3,300 Houses with moderate damage 1,428 Houses with severe damage 533 Health centers affected 15 Education centers affected 16 Government buildings affected 12 Private assets affected 6 Bridges affected 1 • Two emergency collective centres were opened by local authorities to provide comprehensive care to people with damaged homes: Table 1: Emergency collective centres opened The rest of affected families have been taken in by friends and families, while others have chosen to build informal shelters made of plastic sheeting and wood next to their homes to guard their belongings."],
        "country": ["Ecuador"],
        "location": ["Esmeralda", "Tabiazo", "Tachina", "San Mateo", "Vuelta Larga", "Chinca", "Camarones", "Quingüe", "Daule de Muisne", "Súa Atacames", "Cotopaxi", "Guayas", "Imbabura", "Los Ríos", "Manabí", "Pichincha", "Santo Domingo de los Tsáchilas", "Tungurahua"],
        "startYear": 2022,
        "startMonth": 3,
        "startDay": 26,
        "endYear": 2022,
        "endMonth": 4,
        "endDay": 6,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2022-04-12",
        "impactSubtype": "Healthcare Infrastructure",
        "impactValue": 15,
        "impactUnit": "health centers",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["17 issued on 6 April 2022 by SNGRE2, the main damages registered include: Summary of People affected and damages Estimate of people affected People affected 4,097 (819 families) People displaced3 3,705 (741 families) People injured 2 Deceased 1 Estimate of infrastructure damaged Houses with slight damage 3,300 Houses with moderate damage 1,428 Houses with severe damage 533 Health centers affected 15 Education centers affected 16 Government buildings affected 12 Private assets affected 6 Bridges affected 1 • Two emergency collective centres were opened by local authorities to provide comprehensive care to people with damaged homes: Table 1: Emergency collective centres opened The rest of affected families have been taken in by friends and families, while others have chosen to build informal shelters made of plastic sheeting and wood next to their homes to guard their belongings."],
        "country": ["Ecuador"],
        "location": ["Esmeralda", "Tabiazo", "Tachina", "San Mateo", "Vuelta Larga", "Chinca", "Camarones", "Quingüe", "Daule de Muisne", "Súa Atacames", "Cotopaxi", "Guayas", "Imbabura", "Los Ríos", "Manabí", "Pichincha", "Santo Domingo de los Tsáchilas", "Tungurahua"],
        "startYear": 2022,
        "startMonth": 3,
        "startDay": 26,
        "endYear": 2022,
        "endMonth": 4,
        "endDay": 6,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2022-04-12",
        "impactSubtype": "Education Infrastructure",
        "impactValue": 16,
        "impactUnit": "education centers",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["17 issued on 6 April 2022 by SNGRE2, the main damages registered include: Summary of People affected and damages Estimate of people affected People affected 4,097 (819 families) People displaced3 3,705 (741 families) People injured 2 Deceased 1 Estimate of infrastructure damaged Houses with slight damage 3,300 Houses with moderate damage 1,428 Houses with severe damage 533 Health centers affected 15 Education centers affected 16 Government buildings affected 12 Private assets affected 6 Bridges affected 1 • Two emergency collective centres were opened by local authorities to provide comprehensive care to people with damaged homes: Table 1: Emergency collective centres opened The rest of affected families have been taken in by friends and families, while others have chosen to build informal shelters made of plastic sheeting and wood next to their homes to guard their belongings."],
        "country": ["Ecuador"],
        "location": ["Esmeralda", "Tabiazo", "Tachina", "San Mateo", "Vuelta Larga", "Chinca", "Camarones", "Quingüe", "Daule de Muisne", "Súa Atacames", "Cotopaxi", "Guayas", "Imbabura", "Los Ríos", "Manabí", "Pichincha", "Santo Domingo de los Tsáchilas", "Tungurahua"],
        "startYear": 2022,
        "startMonth": 3,
        "startDay": 26,
        "endYear": 2022,
        "endMonth": 4,
        "endDay": 6,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2022-04-12",
        "impactSubtype": "Undefined Infrastructure",
        "impactValue": 12,
        "impactUnit": "government buildings",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["17 issued on 6 April 2022 by SNGRE2, the main damages registered include: Summary of People affected and damages Estimate of people affected People affected 4,097 (819 families) People displaced3 3,705 (741 families) People injured 2 Deceased 1 Estimate of infrastructure damaged Houses with slight damage 3,300 Houses with moderate damage 1,428 Houses with severe damage 533 Health centers affected 15 Education centers affected 16 Government buildings affected 12 Private assets affected 6 Bridges affected 1 • Two emergency collective centres were opened by local authorities to provide comprehensive care to people with damaged homes: Table 1: Emergency collective centres opened The rest of affected families have been taken in by friends and families, while others have chosen to build informal shelters made of plastic sheeting and wood next to their homes to guard their belongings."],
        "country": ["Ecuador"],
        "location": ["Esmeralda", "Tabiazo", "Tachina", "San Mateo", "Vuelta Larga", "Chinca", "Camarones", "Quingüe", "Daule de Muisne", "Súa Atacames", "Cotopaxi", "Guayas", "Imbabura", "Los Ríos", "Manabí", "Pichincha", "Santo Domingo de los Tsáchilas", "Tungurahua"],
        "startYear": 2022,
        "startMonth": 3,
        "startDay": 26,
        "endYear": 2022,
        "endMonth": 4,
        "endDay": 6,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2022-04-12",
        "impactSubtype": "Road Infrastructure",
        "impactValue": 1,
        "impactUnit": "bridges",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": ["17 issued on 6 April 2022 by SNGRE2, the main damages registered include: Summary of People affected and damages Estimate of people affected People affected 4,097 (819 families) People displaced3 3,705 (741 families) People injured 2 Deceased 1 Estimate of infrastructure damaged Houses with slight damage 3,300 Houses with moderate damage 1,428 Houses with severe damage 533 Health centers affected 15 Education centers affected 16 Government buildings affected 12 Private assets affected 6 Bridges affected 1 • Two emergency collective centres were opened by local authorities to provide comprehensive care to people with damaged homes: Table 1: Emergency collective centres opened The rest of affected families have been taken in by friends and families, while others have chosen to build informal shelters made of plastic sheeting and wood next to their homes to guard their belongings."],
        "country": ["Ecuador"],
        "location": ["Esmeralda", "Tabiazo", "Tachina", "San Mateo", "Vuelta Larga", "Chinca", "Camarones", "Quingüe", "Daule de Muisne", "Súa Atacames", "Cotopaxi", "Guayas", "Imbabura", "Los Ríos", "Manabí", "Pichincha", "Santo Domingo de los Tsáchilas", "Tungurahua"],
        "startYear": 2022,
        "startMonth": 3,
        "startDay": 26,
        "endYear": 2022,
        "endMonth": 4,
        "endDay": 6,
        "hazards": ["Earthquake"],
    },
]

In [27]:
df_impact = pd.DataFrame(labelled_impact_reports_dict[iappeal])
df_impact['appealCode'] = iappeal
df_impact.reset_index(inplace=True, drop=True)

#Save impact csv
fn = f"labelled_impact_{iappeal}.csv"
#df_impact.to_csv(DATA_LABELLED+fn, index=False)

# MDRPG008

In [29]:
iappeal = "MDRPG008"
ireport = select_report_by_appealCode(iappeal, reports_to_label).iloc[0]
print(ireport.reportLink)
print_report(iappeal, ireport)

https://adore.ifrc.org/Download.aspx?FileId=219337
MDRPG008: 2018-11-24
EEmmeerrggeennccyy PPllaann ooff AAccttiioonn OOppeerraattiioonn UFipndaal tRe eport Papua New Guinea: Earthquake Papua New Guinea: Earthquake DREF operation Operation n° MDRPG008 Date of Issue: 24 November 2018 Glide number: EQ-2018-000020-PNG Date of disaster: 26 February 2018 Operation start date: 27 February 2018 Operation end date: 27 July 2018 Host National Society(ies): Papua New Guinea Red Cross Operation budget: CHF 209,398 Number of people affected: 270,000 Number of people assisted: 3,000 N° of National Societies involved in the operation: The National Society is working with the International Federation of Red Cross and Red Crescent Societies (IFRC) and International Committee of the Red Cross (ICRC).
N° of other partner organizations involved in the operation: Provincial disaster committees (PDCs), National Disaster Centre (NDC), PNG Disaster Management Team, UN agencies, INGO’s, Exxon Mobile, Oil Sear

In [30]:
labelled_impact_reports_dict[iappeal] = [
    {
        "reportDate": "2018-11-24",
        "impactSubtype": "DREF Allocation & Funding requirements",
        "impactValue": 209398,
        "impactUnit": "CHF",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "EEmmeerrggeennccyy PPllaann ooff AAccttiioonn OOppeerraattiioonn UFipndaal tRe eport Papua New Guinea: Earthquake Papua New Guinea: Earthquake DREF operation Operation n° MDRPG008 Date of Issue: 24 November 2018 Glide number: EQ-2018-000020-PNG Date of disaster: 26 February 2018 Operation start date: 27 February 2018 Operation end date: 27 July 2018 Host National Society(ies): Papua New Guinea Red Cross Operation budget: CHF 209,398 Number of people affected: 270,000 Number of people assisted: 3,000 N° of National Societies involved in the operation: The National Society is working with the International Federation of Red Cross and Red Crescent Societies (IFRC) and International Committee of the Red Cross (ICRC)."
           ],
        "country": ["Papua New Guinea"],
        "location": None,
        "startYear": 2018,
        "startMonth": 2,
        "startDay": 26,
        "endYear": 2018,
        "endMonth": 7,
        "endDay": 27,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2018-11-24",
        "impactSubtype": "Affected people",
        "impactValue": 270000,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "EEmmeerrggeennccyy PPllaann ooff AAccttiioonn OOppeerraattiioonn UFipndaal tRe eport Papua New Guinea: Earthquake Papua New Guinea: Earthquake DREF operation Operation n° MDRPG008 Date of Issue: 24 November 2018 Glide number: EQ-2018-000020-PNG Date of disaster: 26 February 2018 Operation start date: 27 February 2018 Operation end date: 27 July 2018 Host National Society(ies): Papua New Guinea Red Cross Operation budget: CHF 209,398 Number of people affected: 270,000 Number of people assisted: 3,000 N° of National Societies involved in the operation: The National Society is working with the International Federation of Red Cross and Red Crescent Societies (IFRC) and International Committee of the Red Cross (ICRC)."
           ],
        "country": ["Papua New Guinea"],
        "location": None,
        "startYear": 2018,
        "startMonth": 2,
        "startDay": 26,
        "endYear": 2018,
        "endMonth": 7,
        "endDay": 27,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2018-11-24",
        "impactSubtype": "Assisted people",
        "impactValue": 3000,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "EEmmeerrggeennccyy PPllaann ooff AAccttiioonn OOppeerraattiioonn UFipndaal tRe eport Papua New Guinea: Earthquake Papua New Guinea: Earthquake DREF operation Operation n° MDRPG008 Date of Issue: 24 November 2018 Glide number: EQ-2018-000020-PNG Date of disaster: 26 February 2018 Operation start date: 27 February 2018 Operation end date: 27 July 2018 Host National Society(ies): Papua New Guinea Red Cross Operation budget: CHF 209,398 Number of people affected: 270,000 Number of people assisted: 3,000 N° of National Societies involved in the operation: The National Society is working with the International Federation of Red Cross and Red Crescent Societies (IFRC) and International Committee of the Red Cross (ICRC)."
           ],
        "country": ["Papua New Guinea"],
        "location": None,
        "startYear": 2018,
        "startMonth": 2,
        "startDay": 26,
        "endYear": 2018,
        "endMonth": 7,
        "endDay": 27,
        "hazards": ["Earthquake"],
    },    
    {
        "reportDate": "2018-11-24",
        "impactSubtype": "Human Deaths",
        "impactValue": 100 ,
        "impactUnit": "people",
        "impactValuePrecision": "approx",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "Approximately over 100 people dead, with most of confirmed deaths caused by landslides triggered by the earthquake.", 
            "The death toll remains unclear as of today, but it is believed that more than 100 people died, with most of confirmed deaths caused by landslides."
           ],
        "country": ["Papua New Guinea"],
        "location": ["Western Highland", "Southern Highlands", "Hela"],
        "startYear": 2018,
        "startMonth": 2,
        "startDay": 26,
        "endYear": 2018,
        "endMonth": 7,
        "endDay": 27,
        "hazards": ["Earthquake", "Mass movement"],
    },    
    {
        "reportDate": "2018-11-24",
        "impactSubtype": "Affected people",
        "impactValue": None ,
        "impactUnit": "people",
        "impactValuePrecision": "approx",
        "impactValueMin": 270000,
        "impactValueMax": None,
        "annotation": [
            "Earthquake affected over 270,000 people, who experienced the intensity above 6.0 and required humanitarian assistance."
           ],
        "country": ["Papua New Guinea"],
        "location": ["Western Highland", "Southern Highlands", "Hela"],
        "startYear": 2018,
        "startMonth": 2,
        "startDay": 26,
        "endYear": 2018,
        "endMonth": 7,
        "endDay": 27,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2018-11-24",
        "impactSubtype": "Road Infrastructure",
        "impactValue": None ,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "Planning commenced in earnest, once the DREF was approved on 28 February, however the majority of communication and road accessibility was damaged, and accessibility has returned to manageable condition only after being cut off for several weeks."
           ],
        "country": ["Papua New Guinea"],
        "location": ["Western Highland", "Southern Highlands", "Hela"],
        "startYear": 2018,
        "startMonth": 2,
        "startDay": 26,
        "endYear": None,
        "endMonth": None,
        "endDay": None,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2018-11-24",
        "impactSubtype": "Mobility and Access to Transport",
        "impactValue": None ,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "Planning commenced in earnest, once the DREF was approved on 28 February, however the majority of communication and road accessibility was damaged, and accessibility has returned to manageable condition only after being cut off for several weeks.", 
            "The majority of communication and road accessibility has returned to pre-earthquake status after being cut off for several weeks.", 
            "The most remote locations could only be accessed by walking trails and in non-emergency periods lack radio or mobile networks."
           ],
        "country": ["Papua New Guinea"],
        "location": ["Western Highland", "Southern Highlands", "Hela"],
        "startYear": 2018,
        "startMonth": 2,
        "startDay": 26,
        "endYear": None,
        "endMonth": None,
        "endDay": None,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2018-11-24",
        "impactSubtype": "IT and Communication Infrastructure",
        "impactValue": None ,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "The most remote locations could only be accessed by walking trails and in non-emergency periods lack radio or mobile networks."
           ],
        "country": ["Papua New Guinea"],
        "location": ["Western Highland", "Southern Highlands", "Hela"],
        "startYear": 2018,
        "startMonth": 2,
        "startDay": 26,
        "endYear": None,
        "endMonth": None,
        "endDay": None,
        "hazards": ["Earthquake"],
    },
]

In [31]:
df_impact = pd.DataFrame(labelled_impact_reports_dict[iappeal])
df_impact['appealCode'] = iappeal
df_impact.reset_index(inplace=True, drop=True)

#Save impact csv
fn = f"labelled_impact_{iappeal}.csv"
#df_impact.to_csv(DATA_LABELLED+fn, index=False)

# MDRTJ035

In [33]:
iappeal = "MDRTJ035"
ireport = select_report_by_appealCode(iappeal, reports_to_label).iloc[0]
print(ireport.reportLink)
print_report(iappeal, ireport)

https://go-api.ifrc.org/api/DownloadFile/84135/MDRTJ035dfr
MDRTJ035: 2025-12-08
DREF Final Report Tajikistan Earthquake 2023 A destroyed house at Paldorak village, Sughd province.
Photo: The Red Crescent Society of Tajikistan (RCST) Appeal: Total DREF Allocation: Crisis Category: Hazard: MDRTJ035 CHF 188,888 Yellow Earthquake Glide Number: People Aected: People Targeted: EQ-2023-000043-TJK 2,202 people 1,908 people Event Onset: Operation Start Date: Operational End Date: Total Operating Timeframe: Sudden 07-04-2023 31-08-2023 4 months Targeted Areas: Districts of Republican Subordination, Sughd The major donors and partners of the IFRC-DREF include the Red Cross Societies and governments of Australia, Austria, Belgium, Britain, China, Czech, Canada, Denmark, Germany, Ireland, Italy, Japan, Luxembourg, Liechtenstein, Malta, Norway, Spain, Sweden, Switzerland, Thailand, and the Netherlands, as well as DG ECHO, Mondelez Foundation, and other corporate and private donors.
The IFRC, on beha

In [34]:
labelled_impact_reports_dict[iappeal] = [
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "DREF Allocation & Funding requirements",
        "impactValue": 188888,
        "impactUnit": "CHF",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "Photo: The Red Crescent Society of Tajikistan (RCST) Appeal: Total DREF Allocation: Crisis Category: Hazard: MDRTJ035 CHF 188,888 Yellow Earthquake Glide Number: People Aected: People Targeted: EQ-2023-000043-TJK 2,202 people 1,908 people Event Onset: Operation Start Date: Operational End Date: Total Operating Timeframe: Sudden 07-04-2023 31-08-2023 4 months Targeted Areas: Districts of Republican Subordination, Sughd The major donors and partners of the IFRC-DREF include the Red Cross Societies and governments of Australia, Austria, Belgium, Britain, China, Czech, Canada, Denmark, Germany, Ireland, Italy, Japan, Luxembourg, Liechtenstein, Malta, Norway, Spain, Sweden, Switzerland, Thailand, and the Netherlands, as well as DG ECHO, Mondelez Foundation, and other corporate and private donors."
        ],
        "country": ["Tajikistan"],
        "location": ["Districts of Republican Subordination", "Sughd"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 8,
        "endDay": 31,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Affected people",
        "impactValue": 2202,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "Photo: The Red Crescent Society of Tajikistan (RCST) Appeal: Total DREF Allocation: Crisis Category: Hazard: MDRTJ035 CHF 188,888 Yellow Earthquake Glide Number: People Aected: People Targeted: EQ-2023-000043-TJK 2,202 people 1,908 people Event Onset: Operation Start Date: Operational End Date: Total Operating Timeframe: Sudden 07-04-2023 31-08-2023 4 months Targeted Areas: Districts of Republican Subordination, Sughd The major donors and partners of the IFRC-DREF include the Red Cross Societies and governments of Australia, Austria, Belgium, Britain, China, Czech, Canada, Denmark, Germany, Ireland, Italy, Japan, Luxembourg, Liechtenstein, Malta, Norway, Spain, Sweden, Switzerland, Thailand, and the Netherlands, as well as DG ECHO, Mondelez Foundation, and other corporate and private donors."
        ],
        "country": ["Tajikistan"],
        "location": ["Districts of Republican Subordination", "Sughd"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 8,
        "endDay": 31,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Targeted people",
        "impactValue": 1908,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "Photo: The Red Crescent Society of Tajikistan (RCST) Appeal: Total DREF Allocation: Crisis Category: Hazard: MDRTJ035 CHF 188,888 Yellow Earthquake Glide Number: People Aected: People Targeted: EQ-2023-000043-TJK 2,202 people 1,908 people Event Onset: Operation Start Date: Operational End Date: Total Operating Timeframe: Sudden 07-04-2023 31-08-2023 4 months Targeted Areas: Districts of Republican Subordination, Sughd The major donors and partners of the IFRC-DREF include the Red Cross Societies and governments of Australia, Austria, Belgium, Britain, China, Czech, Canada, Denmark, Germany, Ireland, Italy, Japan, Luxembourg, Liechtenstein, Malta, Norway, Spain, Sweden, Switzerland, Thailand, and the Netherlands, as well as DG ECHO, Mondelez Foundation, and other corporate and private donors."
        ],
        "country": ["Tajikistan"],
        "location": ["Districts of Republican Subordination", "Sughd"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 8,
        "endDay": 31,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Injured people",
        "impactValue": 3,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "According to preliminary information, no deaths were reported, however, three residents of the Kuhistoni Mastchoh district were injured."
        ],
        "country": ["Tajikistan"],
        "location": ["Kuhistoni Mastchoh"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": None,
        "endMonth": None,
        "endDay": None,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Residential Buildings",
        "impactValue": 318,
        "impactUnit": "houses damaged",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "Photo credit: RCST Photo credit: RCST Scope and Scale On 28 March 2023, the Tajikistan Governmental Commission identied the scale of damages and losses, which included 318 damaged houses in Kuhistoni Mashoh district of Sughd province."
        ],
        "country": ["Tajikistan"],
        "location": ["Kuhistoni Mastchoh", "Sughd"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 3,
        "endDay": 28,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Residential Buildings",
        "impactValue": 165,
        "impactUnit": "houses damaged",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "The distribution of damaged houses was as follows: - 165 houses were damaged in Paldorak village; - Pakshif village reported 49 damaged houses; - Yarm village documented 23 damaged houses; - 11 damaged houses were identied in Dashti Miyona, 18 in Dehmanoro, 44 in Rogh, 7 in Samjon, and 1 in Khudqi Bolo, all due to the earthquake."
        ],
        "country": ["Tajikistan"],
        "location": ["Paldorak village"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 3,
        "endDay": 28,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Residential Buildings",
        "impactValue": 49,
        "impactUnit": "houses damaged",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "The distribution of damaged houses was as follows: - 165 houses were damaged in Paldorak village; - Pakshif village reported 49 damaged houses; - Yarm village documented 23 damaged houses; - 11 damaged houses were identied in Dashti Miyona, 18 in Dehmanoro, 44 in Rogh, 7 in Samjon, and 1 in Khudqi Bolo, all due to the earthquake."
        ],
        "country": ["Tajikistan"],
        "location": ["Pakshif village"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 3,
        "endDay": 28,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Residential Buildings",
        "impactValue": 23,
        "impactUnit": "houses damaged",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "The distribution of damaged houses was as follows: - 165 houses were damaged in Paldorak village; - Pakshif village reported 49 damaged houses; - Yarm village documented 23 damaged houses; - 11 damaged houses were identied in Dashti Miyona, 18 in Dehmanoro, 44 in Rogh, 7 in Samjon, and 1 in Khudqi Bolo, all due to the earthquake."
        ],
        "country": ["Tajikistan"],
        "location": ["Yarm village"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 3,
        "endDay": 28,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Residential Buildings",
        "impactValue": 11,
        "impactUnit": "houses damaged",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "The distribution of damaged houses was as follows: - 165 houses were damaged in Paldorak village; - Pakshif village reported 49 damaged houses; - Yarm village documented 23 damaged houses; - 11 damaged houses were identied in Dashti Miyona, 18 in Dehmanoro, 44 in Rogh, 7 in Samjon, and 1 in Khudqi Bolo, all due to the earthquake."
        ],
        "country": ["Tajikistan"],
        "location": ["Dashti Miyona"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 3,
        "endDay": 28,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Residential Buildings",
        "impactValue": 18,
        "impactUnit": "houses damaged",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "The distribution of damaged houses was as follows: - 165 houses were damaged in Paldorak village; - Pakshif village reported 49 damaged houses; - Yarm village documented 23 damaged houses; - 11 damaged houses were identied in Dashti Miyona, 18 in Dehmanoro, 44 in Rogh, 7 in Samjon, and 1 in Khudqi Bolo, all due to the earthquake."
        ],
        "country": ["Tajikistan"],
        "location": ["Dehmanoro"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 3,
        "endDay": 28,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Residential Buildings",
        "impactValue": 44,
        "impactUnit": "houses damaged",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "The distribution of damaged houses was as follows: - 165 houses were damaged in Paldorak village; - Pakshif village reported 49 damaged houses; - Yarm village documented 23 damaged houses; - 11 damaged houses were identied in Dashti Miyona, 18 in Dehmanoro, 44 in Rogh, 7 in Samjon, and 1 in Khudqi Bolo, all due to the earthquake."
        ],
        "country": ["Tajikistan"],
        "location": ["Rogh"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 3,
        "endDay": 28,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Residential Buildings",
        "impactValue": 7,
        "impactUnit": "houses damaged",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "The distribution of damaged houses was as follows: - 165 houses were damaged in Paldorak village; - Pakshif village reported 49 damaged houses; - Yarm village documented 23 damaged houses; - 11 damaged houses were identied in Dashti Miyona, 18 in Dehmanoro, 44 in Rogh, 7 in Samjon, and 1 in Khudqi Bolo, all due to the earthquake."
        ],
        "country": ["Tajikistan"],
        "location": ["Samjon"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 3,
        "endDay": 28,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Residential Buildings",
        "impactValue": 1,
        "impactUnit": "houses damaged",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "The distribution of damaged houses was as follows: - 165 houses were damaged in Paldorak village; - Pakshif village reported 49 damaged houses; - Yarm village documented 23 damaged houses; - 11 damaged houses were identied in Dashti Miyona, 18 in Dehmanoro, 44 in Rogh, 7 in Samjon, and 1 in Khudqi Bolo, all due to the earthquake."
        ],
        "country": ["Tajikistan"],
        "location": ["Khudqifi Bolo"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 3,
        "endDay": 28,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Affected Livestock and Animals",
        "impactValue": 18,
        "impactUnit": "large cattle",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "The earthquake furthermore resulted in the destruction of numerous cattle sheds, leading to a signicant loss of both large and small livestock (18 large cattle, 27 small cattle, and two horses)."
        ],
        "country": ["Tajikistan"],
        "location": ["Paldorak", "Yarm", "Pakshif", "Langar", "Kuhistoni Mashoh"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 3,
        "endDay": 28,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Affected Livestock and Animals",
        "impactValue": 27,
        "impactUnit": "small cattle",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "The earthquake furthermore resulted in the destruction of numerous cattle sheds, leading to a signicant loss of both large and small livestock (18 large cattle, 27 small cattle, and two horses)."
        ],
        "country": ["Tajikistan"],
        "location": ["Paldorak", "Yarm", "Pakshif", "Langar", "Kuhistoni Mashoh"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 3,
        "endDay": 28,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Affected Livestock and Animals",
        "impactValue": 2,
        "impactUnit": "horses",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "The earthquake furthermore resulted in the destruction of numerous cattle sheds, leading to a signicant loss of both large and small livestock (18 large cattle, 27 small cattle, and two horses)."
        ],
        "country": ["Tajikistan"],
        "location": ["Paldorak", "Yarm", "Pakshif", "Langar", "Kuhistoni Mashoh"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 3,
        "endDay": 28,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Economy and Livelihood",
        "impactValue": None,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "The earthquake furthermore resulted in the destruction of numerous cattle sheds, leading to a signicant loss of both large and small livestock (18 large cattle, 27 small cattle, and two horses)."
        ],
        "country": ["Tajikistan"],
        "location": ["Paldorak", "Yarm", "Pakshif", "Langar", "Kuhistoni Mashoh"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 3,
        "endDay": 28,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Water, Sanitation, and Hygiene Infrastructure",
        "impactValue": None,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "Furthermore, the earthquake inicted damage upon a 750-meter irrigation canal, which signicantly increased the vulnerability of the local population engaged in agriculture."
        ],
        "country": ["Tajikistan"],
        "location": ["Paldorak", "Yarm", "Pakshif", "Langar", "Kuhistoni Mashoh"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 3,
        "endDay": 28,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Crop Production and Forestry",
        "impactValue": None,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "Furthermore, the earthquake inicted damage upon a 750-meter irrigation canal, which signicantly increased the vulnerability of the local population engaged in agriculture."
        ],
        "country": ["Tajikistan"],
        "location": ["Paldorak", "Yarm", "Pakshif", "Langar", "Kuhistoni Mashoh"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 3,
        "endDay": 28,
        "hazards": ["Earthquake"],
    },
    {
        "reportDate": "2025-12-08",
        "impactSubtype": "Access to Water, Sanitation, and Hygiene",
        "impactValue": None,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "annotation": [
            "Furthermore, the earthquake inicted damage upon a 750-meter irrigation canal, which signicantly increased the vulnerability of the local population engaged in agriculture.", 
            "This canal also served as a source of drinking water."
        ],
        "country": ["Tajikistan"],
        "location": ["Paldorak", "Yarm", "Pakshif", "Langar", "Kuhistoni Mashoh"],
        "startYear": 2023,
        "startMonth": 3,
        "startDay": 23,
        "endYear": 2023,
        "endMonth": 3,
        "endDay": 28,
        "hazards": ["Earthquake"],
    },
]

In [35]:
df_impact = pd.DataFrame(labelled_impact_reports_dict[iappeal])
df_impact['appealCode'] = iappeal
df_impact.reset_index(inplace=True, drop=True)

#Save impact csv
fn = f"labelled_impact_{iappeal}.csv"
#df_impact.to_csv(DATA_LABELLED+fn, index=False)

# MDRPK018

In [8]:
iappeal = "MDRPK018"
ireport = select_report_by_appealCode(iappeal, reports_to_label).iloc[0]
# print(ireport.reportLink)
print_report(iappeal, ireport)

MDRPK018: 2020-11-30
Final Report Pakistan: Severe winter DREF operation Operation n° MDRPK018 Date of Issue: 30 November 2020 Glide number n° CW-2020-000027-PAK Operation start date: 5 February 2020 Operation end date: 31 August 2020 Operation budget: CHF 315,292 Host National Society: Pakistan Red Crescent Society (PRCS) Number of people affected: 800,000 Number of people assisted: 6,291 people (967 households) Red Cross Red Crescent Movement partners currently actively involved in the operation: The International Federation of Red Cross and Red Crescent Societies (IFRC) and International Committee of the Red Cross (ICRC) were actively involved in supporting Pakistan Red Crescent Society’s (PRCS) response for severe snowfall in Azad Jammu and Kashmir (AJK) and Balochistan.
Other partner organizations involved in the operation: State Disaster Management Authority (SDMA), National Disaster Management Authority (NDMA) and District Disaster Management Unit (DDMU) were the Government auth

In [13]:
labelled_impact_reports_dict[iappeal] = [
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "DREF Allocation & Funding requirements",
        "impactValue": 315292,
        "impactUnit": "CHF",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan", "Azad Jammu and Kashmir"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 8,
        "endDay": 31,
        "hazards": ["Flood", "Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            "Final Report Pakistan: Severe winter DREF operation Operation n° MDRPK018 Date of Issue: 30 November 2020 Glide number n° CW-2020-000027-PAK Operation start date: 5 February 2020 Operation end date: 31 August 2020 Operation budget: CHF 315,292 Host National Society: Pakistan Red Crescent Society (PRCS) Number of people affected: 800,000 Number of people assisted: 6,291 people (967 households) Red Cross Red Crescent Movement partners currently actively involved in the operation: The International Federation of Red Cross and Red Crescent Societies (IFRC) and International Committee of the Red Cross (ICRC) were actively involved in supporting Pakistan Red Crescent Society’s (PRCS) response for severe snowfall in Azad Jammu and Kashmir (AJK) and Balochistan.",
        ],
    }, 
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Assisted People",
        "impactValue": 6291,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan", "Azad Jammu and Kashmir"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 8,
        "endDay": 31,
        "hazards": ["Flood", "Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            "Final Report Pakistan: Severe winter DREF operation Operation n° MDRPK018 Date of Issue: 30 November 2020 Glide number n° CW-2020-000027-PAK Operation start date: 5 February 2020 Operation end date: 31 August 2020 Operation budget: CHF 315,292 Host National Society: Pakistan Red Crescent Society (PRCS) Number of people affected: 800,000 Number of people assisted: 6,291 people (967 households) Red Cross Red Crescent Movement partners currently actively involved in the operation: The International Federation of Red Cross and Red Crescent Societies (IFRC) and International Committee of the Red Cross (ICRC) were actively involved in supporting Pakistan Red Crescent Society’s (PRCS) response for severe snowfall in Azad Jammu and Kashmir (AJK) and Balochistan.",
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Affected People",
        "impactValue": 800000,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan", "Azad Jammu and Kashmir"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 8,
        "endDay": 31,
        "hazards": ["Flood", "Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            "Final Report Pakistan: Severe winter DREF operation Operation n° MDRPK018 Date of Issue: 30 November 2020 Glide number n° CW-2020-000027-PAK Operation start date: 5 February 2020 Operation end date: 31 August 2020 Operation budget: CHF 315,292 Host National Society: Pakistan Red Crescent Society (PRCS) Number of people affected: 800,000 Number of people assisted: 6,291 people (967 households) Red Cross Red Crescent Movement partners currently actively involved in the operation: The International Federation of Red Cross and Red Crescent Societies (IFRC) and International Committee of the Red Cross (ICRC) were actively involved in supporting Pakistan Red Crescent Society’s (PRCS) response for severe snowfall in Azad Jammu and Kashmir (AJK) and Balochistan.",
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Human Deaths",
        "impactValue": None,
        "impactUnit": "people",
        "impactValuePrecision": "approx",
        "impactValueMin": 107,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan", "Azad Jammu and Kashmir", "Khyber Pakhtunkhwa", "Gilgit Baltistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature"],
        "annotation": [
            'According to the National Disaster Management Authority (NDMA) situation report published on 23 January 20201, at least 107 people died because of the severe weather conditions.',
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Human Deaths",
        "impactValue": 21,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature"],
        "annotation": [
            'Among them, 21 fatalities were in Balochistan, five in Khyber Pakhtunkhwa (KP), two in Gilgit Baltistan (GB) and 79 in AJK.',
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Human Deaths",
        "impactValue": 5,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Khyber Pakhtunkhwa"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature"],
        "annotation": [
            'Among them, 21 fatalities were in Balochistan, five in Khyber Pakhtunkhwa (KP), two in Gilgit Baltistan (GB) and 79 in AJK.',
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Human Deaths",
        "impactValue": 2,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Gilgit Baltistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature"],
        "annotation": [
            'Among them, 21 fatalities were in Balochistan, five in Khyber Pakhtunkhwa (KP), two in Gilgit Baltistan (GB) and 79 in AJK.',
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Human Deaths",
        "impactValue": 79,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Azad Jammu and Kashmir"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature"],
        "annotation": [
            'Among them, 21 fatalities were in Balochistan, five in Khyber Pakhtunkhwa (KP), two in Gilgit Baltistan (GB) and 79 in AJK.',
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Road Infrastructure",
        "impactValue": None,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Quetta-Sibi", "Quetta-Karachi", "Quetta-Zhob"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1, 
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Other storm"],
        "annotation": [
            'The highways between Quetta-Sibi, Quetta-Karachi and Quetta-Zhob were also blocked.',
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Affected People",
        "impactValue": 1000000,
        "impactUnit": "people",
        "impactValuePrecision": "approx",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan", "Azad Jammu and Kashmir", "Khyber Pakhtunkhwa", "Gilgit Baltistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature"],
        "annotation": [
            'Meanwhile, the United Nations Office for the 1 Coordination of Humanitarian Affairs (UNOCHA) reported that around 1 million people had been affected by the cold wave (around 140,000 families)2.'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Human Deaths",
        "impactValue": 79,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Neelum"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Other storm", "Flood", "Mass movement"],
        "annotation": [
            'In Kashmir, the worst affected district was Neelum (extreme north district of AJK) due to heavy snowfall, rain and avalanche created havoc, whereby 79 people died, and more than 91 houses were destroyed.'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Residential Buildings",
        "impactValue": 91,
        "impactUnit": "destroyed buildings",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Neelum"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Other storm", "Flood", "Mass movement"],
        "annotation": [
            'In Kashmir, the worst affected district was Neelum (extreme north district of AJK) due to heavy snowfall, rain and avalanche created havoc, whereby 79 people died, and more than 91 houses were destroyed.'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Affected People",
        "impactValue": 916370,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Neelum"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'The total population affected by the avalanches in district Neelum were 910 households (6,370 people), with another 3,134 households (21,938 people) indirectly affected by the heavy snow.'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Affected People",
        "impactValue": 21938,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Neelum"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'The total population affected by the avalanches in district Neelum were 910 households (6,370 people), with another 3,134 households (21,938 people) indirectly affected by the heavy snow.'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Human Deaths",
        "impactValue": 2,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Astore valley"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Evacuation of dead bodies from the snow, (Photos: PRCS) Astore valley received 100 years record-breaking snowfall in some areas, as reported by media and Gilgit Baltistan Disaster Management Authority (GBDMA).',
            'Two people lost their lives due to an avalanche, while four were injured.'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Injured People",
        "impactValue": 4,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Astore valley"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Evacuation of dead bodies from the snow, (Photos: PRCS) Astore valley received 100 years record-breaking snowfall in some areas, as reported by media and Gilgit Baltistan Disaster Management Authority (GBDMA).',
            'Two people lost their lives due to an avalanche, while four were injured.'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Mobility and Access to Transport",
        "impactValue": None,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Northern valleys", "Balochistan", "Azad Jammu and Kashmir", "Khyber Pakhtunkhwa", "Gilgit Baltistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'The heavy snowfall paralyzed life in the region, with residents in northern valleys restricted to their homes.'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Economy and Livelihood",
        "impactValue": None,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Northern valleys", "Balochistan", "Azad Jammu and Kashmir", "Khyber Pakhtunkhwa", "Gilgit Baltistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'The heavy snowfall paralyzed life in the region, with residents in northern valleys restricted to their homes.'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Mobility and Access to Transport",
        "impactValue": None,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan", "Azad Jammu and Kashmir", "Khyber Pakhtunkhwa", "Gilgit Baltistan", "Neelum", "Leepa"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Gilgit Baltistan authorities mobilized all their resources to open the blocked roads and to assist the people in need.', 
            'The roads and telecommunication networks were severely affected in AJK, Balochistan, KP, GB and adjoining areas owing to the heavy downpour in plains and snowfall on mountains.', 
            'As a result of heavy rain and snowfall, the upper reaches of Neelum and Leepa and some mountainous tops of AJK were disconnected from the rest of the country through land routes.', 
            'Furthermore, since local people in this region make their living through daily wage jobs, livestock rearing, farming and small businesses, blockage of pathways and roads directly affected their livelihoods.'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Road Infrastructure",
        "impactValue": None,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan", "Azad Jammu and Kashmir", "Khyber Pakhtunkhwa", "Gilgit Baltistan", "Neelum", "Leepa"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Gilgit Baltistan authorities mobilized all their resources to open the blocked roads and to assist the people in need.', 
            'The roads and telecommunication networks were severely affected in AJK, Balochistan, KP, GB and adjoining areas owing to the heavy downpour in plains and snowfall on mountains.', 
            'As a result of heavy rain and snowfall, the upper reaches of Neelum and Leepa and some mountainous tops of AJK were disconnected from the rest of the country through land routes.', 
            'Furthermore, since local people in this region make their living through daily wage jobs, livestock rearing, farming and small businesses, blockage of pathways and roads directly affected their livelihoods.'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Economy and Livelihood",
        "impactValue": None,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan", "Azad Jammu and Kashmir", "Khyber Pakhtunkhwa", "Gilgit Baltistan", "Neelum", "Leepa"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Furthermore, since local people in this region make their living through daily wage jobs, livestock rearing, farming and small businesses, blockage of pathways and roads directly affected their livelihoods.'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Crop Production and Forestry",
        "impactValue": None,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan", "Azad Jammu and Kashmir", "Khyber Pakhtunkhwa", "Gilgit Baltistan", "Neelum", "Leepa"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Furthermore, since local people in this region make their living through daily wage jobs, livestock rearing, farming and small businesses, blockage of pathways and roads directly affected their livelihoods.'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Affected Livestock and Animals",
        "impactValue": None,
        "impactUnit": None,
        "impactValuePrecision": None,
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan", "Azad Jammu and Kashmir", "Khyber Pakhtunkhwa", "Gilgit Baltistan", "Neelum", "Leepa"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Furthermore, since local people in this region make their living through daily wage jobs, livestock rearing, farming and small businesses, blockage of pathways and roads directly affected their livelihoods.'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Human Deaths",
        "impactValue": 79,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Azad Jammu and Kashmir"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Injured People",
        "impactValue": 63,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Azad Jammu and Kashmir"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Residential Buildings",
        "impactValue": 91,
        "impactUnit": "destroyed buildings",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Azad Jammu and Kashmir"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Residential Buildings",
        "impactValue": 202,
        "impactUnit": "damaged buildings",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Azad Jammu and Kashmir"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Undefined Infrastructure",
        "impactValue": 1,
        "impactUnit": "mosque",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Azad Jammu and Kashmir"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Undefined Infrastructure",
        "impactValue": 22,
        "impactUnit": "shops",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Azad Jammu and Kashmir"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Mobility and Access to Transport",
        "impactValue": 7,
        "impactUnit": "Light Transport Vehicle",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Azad Jammu and Kashmir"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Mobility and Access to Transport",
        "impactValue": 3,
        "impactUnit": "motorcycles",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Azad Jammu and Kashmir"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Road Infrastructure",
        "impactValue": 3,
        "impactUnit": "motorcycles",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Neelum", "Lawat", "Tao Butt", "Leepa"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Human Deaths",
        "impactValue": 21,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Injured People",
        "impactValue": 24,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Residential Buildings",
        "impactValue": 148,
        "impactUnit": "destroyed buildings",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Residential Buildings",
        "impactValue": 1062,
        "impactUnit": "damaged buildings",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Road Infrastructure",
        "impactValue": 1,
        "impactUnit": "bridge",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Undefined Infrastructure",
        "impactValue": 1,
        "impactUnit": "mosque",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Mobility and Access to Transport",
        "impactValue": 9,
        "impactUnit": "Light Transport Vehicles",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Mobility and Access to Transport",
        "impactValue": 3,
        "impactUnit": "motorcycles",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Affected Livestock and Animals",
        "impactValue": 29,
        "impactUnit": "livestock",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Balochistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Human Deaths",
        "impactValue": 5,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Khyber Pakhtunkhwa"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Injured People",
        "impactValue": 13,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Khyber Pakhtunkhwa"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Residential Buildings",
        "impactValue": 31,
        "impactUnit": "destroyed buildings",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Khyber Pakhtunkhwa"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Residential Buildings",
        "impactValue": 33,
        "impactUnit": "damaged buildings",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Khyber Pakhtunkhwa"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Human Deaths",
        "impactValue": 2,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Gilgit-Baltistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Injured People",
        "impactValue": 4,
        "impactUnit": "people",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Gilgit-Baltistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Residential Buildings",
        "impactValue": 3,
        "impactUnit": "destroyed buildings",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Gilgit-Baltistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
    {
        "reportDate": ireport.reportDate,
        "impactSubtype": "Residential Buildings",
        "impactValue": 3,
        "impactUnit": "damaged buildings",
        "impactValuePrecision": "exact",
        "impactValueMin": None,
        "impactValueMax": None,
        "country": ["Pakistan"],
        "location": ["Gilgit-Baltistan"],
        "startYear": 2020,
        "startMonth": 1,
        "startDay": 1,
        "endYear": 2020,
        "endMonth": 1,
        "endDay": 23,
        "hazards": ["Extreme cold temperature", "Mass movement", "Other storm"],
        "annotation": [
            'Table 1: Summary of deaths, injuries and losses (Source: NDMA Sit-Rep – 23 January 2020) Houses damaged Provinces Deaths Injured Others Fully Partially One mosque, 22 shops, 7 vehicles Light Transport Vehicle (LTV), AJK 79 63 91 202 3 Motorcycles damaged.',
            'In upper Neelum (Lawat to Tao Butt) Leepa roads blocked.',
            'One bridge collapsed, one mosque completely collapsed, 9 Light Balochistan 21 24 148 1,062 Transport Vehicles (LTV), 3 motorcycles damaged and 29 livestock perished.',
            'KP 5 13 31 33 GB 2 4 3 3 100 years record snow in some areas (Media and GBDMA) Total 107 104 273 1,300'
        ],
    },
]

In [14]:
df_impact = pd.DataFrame(labelled_impact_reports_dict[iappeal])
df_impact['appealCode'] = iappeal
df_impact.reset_index(inplace=True, drop=True)

#Save impact csv
fn = f"labelled_impact_{iappeal}.csv"
#df_impact.to_csv(DATA_LABELLED+fn, index=False)

# Save all reports

In [41]:
labelled_reports_df = report_dict_to_df(labelled_impact_reports_dict)
labelled_reports_df.dropna(how="all").appealCode.unique()
labelled_reports_df["reportDate"] = pd.to_datetime(labelled_reports_df["reportDate"])

/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/labelling_helpers.py:118: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat(df_list)


In [42]:
labelled_reports_df

,reportDate,impactSubtype,impactValue,impactUnit,impactValuePrecision,impactValueMin,impactValueMax,annotation,country,location,startYear,startMonth,startDay,endYear,endMonth,endDay,hazards,appealCode,comments
0,2019-03-14,Affected People,1381000.0,people,exact,NaN,NaN,[SITUATION ANALYSIS Description of the disaste...,[China],"[Deyang, Mianyang, Guangyuan]",2018.0,7.0,7.0,2018.0,7.0,13.0,"[Flood, Tropical storm]",MDRCN006,NaN
1,2019-03-14,Human Deaths,3.0,people,exact,NaN,NaN,[SITUATION ANALYSIS Description of the disaste...,[China],"[Deyang, Mianyang, Guangyuan]",2018.0,7.0,7.0,2018.0,7.0,13.0,"[Flood, Tropical storm]",MDRCN006,NaN
2,2019-03-14,Displaced People,222000.0,people,exact,NaN,NaN,[SITUATION ANALYSIS Description of the disaste...,[China],"[Deyang, Mianyang, Guangyuan]",2018.0,7.0,7.0,2018.0,7.0,13.0,"[Flood, Tropical storm]",MDRCN006,NaN
3,2019-03-14,Residential Buildings,NaN,houses collapsed,approx,900.0,NaN,[SITUATION ANALYSIS Description of the disaste...,[China],"[Deyang, Mianyang, Guangyuan]",2018.0,7.0,7.0,2018.0,7.0,13.0,"[Flood, Tropical storm]",MDRCN006,NaN
4,2019-03-14,Residential Buildings,NaN,houses damaged,approx,29000.0,NaN,[SITUATION ANALYSIS Description of the disaste...,[China],"[Deyang, Mianyang, Guangyuan]",2018.0,7.0,7.0,2018.0,7.0,13.0,"[Flood, Tropical storm]",MDRCN006,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273,2024-08-30,Affected People,125.0,households affected,exact,NaN,NaN,[Kanara Town Council also faced significant da...,[Uganda],[Kanara Town Council],2024.0,8.0,NaN,2024.0,8.0,NaN,[Flood],MDRUG050,NaN
274,2024-08-30,Displaced People,66.0,people,exact,NaN,NaN,[Kanara Town Council also faced significant da...,[Uganda],[Kanara Town Council],2024.0,8.0,NaN,2024.0,8.0,NaN,[Flood],MDRUG050,NaN
275,2024-08-30,Residential Buildings,1705.0,households fully destroyed,exact,NaN,NaN,"[Overall, 1,705 households were fully destroye...",[Uganda],[Ntoroko District],2024.0,8.0,NaN,2024.0,8.0,NaN,[Flood],MDRUG050,NaN
276,2024-08-30,Residential Buildings,1498.0,households partially destroyed,exact,NaN,NaN,"[Overall, 1,705 households were fully destroye...",[Uganda],[Ntoroko District],2024.0,8.0,NaN,2024.0,8.0,NaN,[Flood],MDRUG050,NaN


In [45]:
# save
filename = f"labelled_reports_impacts_laura_nogaps_v{dt.datetime.now().strftime('%d%m%y')}"
labelled_reports_df.to_csv(DATA_LABELLED / (filename + ".csv"), index=False)